In [1]:
!pip install opentelemetry-api==1.20.0 opentelemetry-sdk==1.20.0 opentelemetry-exporter-otlp==1.20.0
!pip install mlflow

  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.31.1
    Uninstalling protobuf-6.31.1:
      Successfully uninstalled protobuf-6.31.1
  Attempting uninstall: opentelemetry-semantic-conventions━━━━━━━━  1/13 [protobuf]
    Found existing installation: opentelemetry-semantic-conventions 0.60b0/13 [protobuf]
    Uninstalling opentelemetry-semantic-conventions-0.60b0:━━━  1/13 [protobuf]
      Successfully uninstalled opentelemetry-semantic-conventions-0.60b0 1/13 [protobuf]
  Attempting uninstall: importlib-metadata━━━━━━━━━━━━━━━━━━━━  1/13 [protobuf]
    Found existing installation: importlib_metadata 8.7.0━━━━━  1/13 [protobuf]
    Uninstalling importlib_metadata-8.7.0:━━━━━━━━━━━━━━━━━━━━  1/13 [protobuf]
      Successfully uninstalled importlib_metadata-8.7.0━━━━━━━  1/13 [protobuf]
  Attempting uninstall: opentelemetry-proto━━━━━━━━━━━━━━━━━━━  1/13 [protobuf]
    Found existing installation: opentelemetry-proto 1.39.0━━━  1/13 [protobuf]
    Uninstallin

In [14]:
from opentelemetry import metrics
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.exporter.otlp.proto.grpc.metric_exporter import OTLPMetricExporter
from opentelemetry.sdk.metrics.export import PeriodicExportingMetricReader
import time

# OTLP gRPC exporter → porta 4317
exporter = OTLPMetricExporter(
    endpoint="mlops-collector-collector.monitoring-tools.svc.cluster.local:8889",
    insecure=True  # IMPORTANTISSIMO in OpenShift
)

reader = PeriodicExportingMetricReader(exporter)
provider = MeterProvider(metric_readers=[reader])
metrics.set_meter_provider(provider)

meter = metrics.get_meter("training-metrics")

# Una metriche molto semplice: counter che possiamo usare come gauge
accuracy = meter.create_up_down_counter(
    name="training_accuracy",
    unit="1",
    description="accuracy fittizia"
)

print("Invio metriche al Collector gRPC...")

last_value = 0

def set_accuracy(val):
    global last_value
    accuracy.add(val - last_value)  # simula una gauge
    last_value = val

# invio alcuni valori di prova:
for v in [0.1, 0.3, 0.6, 0.2, 0.9]:
    print("accuracy =", v)
    set_accuracy(v)
    time.sleep(3)

print("Fatto.")


Overriding of current MeterProvider is not allowed
An instrument with name training_accuracy, type UpDownCounter, unit 1 and description accuracy fittizia has been created already.


Invio metriche al Collector gRPC...
accuracy = 0.1
accuracy = 0.3
accuracy = 0.6
accuracy = 0.2
accuracy = 0.9
Fatto.


Transient error StatusCode.UNAVAILABLE encountered while exporting metrics to mlops-collector-collector.monitoring-tools.svc.cluster.local:8889, retrying in 1s.
Transient error StatusCode.UNAVAILABLE encountered while exporting metrics to mlops-collector-collector.monitoring-tools.svc.cluster.local:8889, retrying in 2s.
Transient error StatusCode.UNAVAILABLE encountered while exporting metrics to mlops-collector-collector.monitoring-tools.svc.cluster.local:8889, retrying in 4s.
Transient error StatusCode.UNAVAILABLE encountered while exporting metrics to mlops-collector-collector.monitoring-tools.svc.cluster.local:8889, retrying in 8s.
Transient error StatusCode.UNAVAILABLE encountered while exporting metrics to mlops-collector-collector.monitoring-tools.svc.cluster.local:8889, retrying in 16s.
Transient error StatusCode.UNAVAILABLE encountered while exporting metrics to mlops-collector-collector.monitoring-tools.svc.cluster.local:8889, retrying in 32s.


In [8]:
import random
import time

from opentelemetry import metrics
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.exporter.otlp.proto.grpc.metric_exporter import OTLPMetricExporter
from opentelemetry.sdk.metrics.export import PeriodicExportingMetricReader


# -----------------------------
# CONFIGURAZIONE EXPORTER OTLP
# -----------------------------
exporter = OTLPMetricExporter(
    endpoint="mlops-collector-collector.monitoring-tools.svc.cluster.local:8889",
    insecure=True
)

reader = PeriodicExportingMetricReader(exporter)
provider = MeterProvider(metric_readers=[reader])
metrics.set_meter_provider(provider)
meter = metrics.get_meter("ml-training-otel")

print("OpenTelemetry configurato per invio al Collector ✓")


# ---------------------------------------
# CREAZIONE DELLE METRICHE COME MLflow 🔥
# ---------------------------------------

def create_gauge(name, desc):
    """Usiamo un UpDownCounter per simulare una gauge."""
    counter = meter.create_up_down_counter(
        name=name,
        unit="1",
        description=desc
    )
    return counter, 0.0


# Metriche classiche di classificazione
accuracy, last_accuracy = create_gauge("accuracy", "training accuracy")
precision, last_precision = create_gauge("precision", "training precision")
recall, last_recall = create_gauge("recall", "training recall")
f1_score, last_f1 = create_gauge("f1_score", "training f1")
roc_auc, last_roc = create_gauge("roc_auc", "roc auc score")

# Metriche di training
training_loss, last_train_loss = create_gauge("training_loss", "training loss")
validation_loss, last_val_loss = create_gauge("validation_loss", "validation loss")
learning_rate, last_lr = create_gauge("learning_rate", "learning rate")

# metrica random
random_metric, last_random_metric = create_gauge("random_metric", "random metric")

# step counter
training_step = meter.create_up_down_counter(
    name="training_step",
    unit="1",
    description="training step"
)

current_step = 0


def set_metric(counter, last, new_value):
    """Simula gauge tramite differenza."""
    diff = new_value - last
    counter.add(diff)
    return new_value


# ---------------------------
# LOOP DI TRAINING SIMULATO
# ---------------------------
for step in range(5):
    print(f"\n--- STEP {step} ---")

    # Classificazione simulate
    last_accuracy = set_metric(accuracy, last_accuracy, random.random())
    last_precision = set_metric(precision, last_precision, random.random())
    last_recall = set_metric(recall, last_recall, random.random())
    last_f1 = set_metric(f1_score, last_f1, random.random())
    last_roc = set_metric(roc_auc, last_roc, random.random())

    # metriche generiche
    last_train_loss = set_metric(training_loss, last_train_loss, random.random())
    last_val_loss = set_metric(validation_loss, last_val_loss, random.random())
    last_lr = set_metric(learning_rate, last_lr, 0.001)

    # random
    last_random_metric = set_metric(random_metric, last_random_metric, random.random())

    # step
    training_step.add(1)

    print("Metriche inviate allo stack OpenTelemetry ✓")
    time.sleep(3)

print("\nTraining completato ✔️ Le metriche ora arrivano in:")
print("- Collector → /metrics")
print("- Prometheus → Query")
print("- Grafana → Dashboard")

Overriding of current MeterProvider is not allowed
An instrument with name accuracy, type UpDownCounter, unit 1 and description training accuracy has been created already.
An instrument with name precision, type UpDownCounter, unit 1 and description training precision has been created already.
An instrument with name recall, type UpDownCounter, unit 1 and description training recall has been created already.
An instrument with name f1_score, type UpDownCounter, unit 1 and description training f1 has been created already.
An instrument with name roc_auc, type UpDownCounter, unit 1 and description roc auc score has been created already.
An instrument with name training_loss, type UpDownCounter, unit 1 and description training loss has been created already.
An instrument with name validation_loss, type UpDownCounter, unit 1 and description validation loss has been created already.
An instrument with name learning_rate, type UpDownCounter, unit 1 and description learning rate has been crea

OpenTelemetry configurato per invio al Collector ✓

--- STEP 0 ---
Metriche inviate allo stack OpenTelemetry ✓

--- STEP 1 ---
Metriche inviate allo stack OpenTelemetry ✓

--- STEP 2 ---
Metriche inviate allo stack OpenTelemetry ✓

--- STEP 3 ---
Metriche inviate allo stack OpenTelemetry ✓

--- STEP 4 ---
Metriche inviate allo stack OpenTelemetry ✓

Training completato ✔️ Le metriche ora arrivano in:
- Collector → /metrics
- Prometheus → Query
- Grafana → Dashboard


Transient error StatusCode.UNAVAILABLE encountered while exporting metrics to mlops-collector-collector.monitoring-tools.svc.cluster.local:8889, retrying in 1s.
Transient error StatusCode.UNAVAILABLE encountered while exporting metrics to mlops-collector-collector.monitoring-tools.svc.cluster.local:8889, retrying in 2s.
Transient error StatusCode.UNAVAILABLE encountered while exporting metrics to mlops-collector-collector.monitoring-tools.svc.cluster.local:8889, retrying in 4s.
Transient error StatusCode.UNAVAILABLE encountered while exporting metrics to mlops-collector-collector.monitoring-tools.svc.cluster.local:8889, retrying in 8s.
Transient error StatusCode.UNAVAILABLE encountered while exporting metrics to mlops-collector-collector.monitoring-tools.svc.cluster.local:8889, retrying in 16s.
Transient error StatusCode.UNAVAILABLE encountered while exporting metrics to mlops-collector-collector.monitoring-tools.svc.cluster.local:8889, retrying in 32s.
Transient error StatusCode.UNAVA

In [9]:
import random
import time
import uuid

from opentelemetry import metrics
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.exporter.otlp.proto.grpc.metric_exporter import OTLPMetricExporter
from opentelemetry.sdk.metrics.export import PeriodicExportingMetricReader


# -----------------------------
# CONFIGURAZIONE EXPORTER OTLP
# -----------------------------
exporter = OTLPMetricExporter(
    endpoint="mlops-collector-collector.monitoring-tools.svc.cluster.local:8889",
    insecure=True
)

reader = PeriodicExportingMetricReader(exporter)
provider = MeterProvider(metric_readers=[reader])
metrics.set_meter_provider(provider)
meter = metrics.get_meter("ml-training-otel")

print("OpenTelemetry configurato ✓")

# -----------------------------
# EXPERIMENT E RUN COME MLflow
# -----------------------------
experiment_name = "ds_dashboard_demo"
run_id = str(uuid.uuid4())[:8]  # simula l'MLflow run_uuid

print(f"Experiment = {experiment_name}")
print(f"Run ID     = {run_id}")

# -----------------------------
# FUNZIONE PER CREARE GAUGE
# -----------------------------
def create_gauge(name, desc):
    counter = meter.create_up_down_counter(
        name=name,
        unit="1",
        description=desc
    )
    return counter, 0.0


# Metriche principali
accuracy, last_accuracy = create_gauge("accuracy", "training accuracy")
precision, last_precision = create_gauge("precision", "training precision")
recall, last_recall = create_gauge("recall", "training recall")
f1_score, last_f1 = create_gauge("f1_score", "training f1")
roc_auc, last_roc = create_gauge("roc_auc", "roc auc metric")

training_loss, last_train_loss = create_gauge("training_loss", "training loss")
validation_loss, last_val_loss = create_gauge("validation_loss", "validation loss")
learning_rate, last_lr = create_gauge("learning_rate", "learning rate")

random_metric, last_random_metric = create_gauge("random_metric", "random metric")


# Counter dello step
training_step = meter.create_up_down_counter(
    name="training_step",
    unit="1",
    description="training step"
)

current_step = 0


def set_metric(counter, last, new_value, step):
    """
    Aggiunge un valore con LABELS run / experiment / step.
    """
    diff = new_value - last
    counter.add(
        diff,
        {
            "experiment": experiment_name,
            "run": run_id,
            "step": step
        }
    )
    return new_value


# -----------------------------
# SIMULAZIONE TRAINING
# -----------------------------
for step in range(5):
    print(f"\n--- STEP {step} ---")

    last_accuracy = set_metric(accuracy, last_accuracy, random.random(), step)
    last_precision = set_metric(precision, last_precision, random.random(), step)
    last_recall = set_metric(recall, last_recall, random.random(), step)
    last_f1 = set_metric(f1_score, last_f1, random.random(), step)
    last_roc = set_metric(roc_auc, last_roc, random.random(), step)

    last_train_loss = set_metric(training_loss, last_train_loss, random.random(), step)
    last_val_loss = set_metric(validation_loss, last_val_loss, random.random(), step)
    last_lr = set_metric(learning_rate, last_lr, 0.001, step)

    last_random_metric = set_metric(random_metric, last_random_metric, random.random(), step)

    training_step.add(1, {"experiment": experiment_name, "run": run_id})

    print("Metriche inviate ✓")
    time.sleep(3)

print("\nTraining completato. Metriche disponibili in Prometheus e Grafana!")


Overriding of current MeterProvider is not allowed
An instrument with name accuracy, type UpDownCounter, unit 1 and description training accuracy has been created already.
An instrument with name precision, type UpDownCounter, unit 1 and description training precision has been created already.
An instrument with name recall, type UpDownCounter, unit 1 and description training recall has been created already.
An instrument with name f1_score, type UpDownCounter, unit 1 and description training f1 has been created already.
An instrument with name training_loss, type UpDownCounter, unit 1 and description training loss has been created already.
An instrument with name validation_loss, type UpDownCounter, unit 1 and description validation loss has been created already.
An instrument with name learning_rate, type UpDownCounter, unit 1 and description learning rate has been created already.
An instrument with name random_metric, type UpDownCounter, unit 1 and description random metric has bee

OpenTelemetry configurato ✓
Experiment = ds_dashboard_demo
Run ID     = 14ecbb98

--- STEP 0 ---
Metriche inviate ✓

--- STEP 1 ---
Metriche inviate ✓

--- STEP 2 ---
Metriche inviate ✓

--- STEP 3 ---
Metriche inviate ✓

--- STEP 4 ---
Metriche inviate ✓

Training completato. Metriche disponibili in Prometheus e Grafana!


Transient error StatusCode.UNAVAILABLE encountered while exporting metrics to mlops-collector-collector.monitoring-tools.svc.cluster.local:8889, retrying in 1s.
Transient error StatusCode.UNAVAILABLE encountered while exporting metrics to mlops-collector-collector.monitoring-tools.svc.cluster.local:8889, retrying in 2s.
Transient error StatusCode.UNAVAILABLE encountered while exporting metrics to mlops-collector-collector.monitoring-tools.svc.cluster.local:8889, retrying in 4s.
Transient error StatusCode.UNAVAILABLE encountered while exporting metrics to mlops-collector-collector.monitoring-tools.svc.cluster.local:8889, retrying in 8s.
Transient error StatusCode.UNAVAILABLE encountered while exporting metrics to mlops-collector-collector.monitoring-tools.svc.cluster.local:8889, retrying in 16s.
Transient error StatusCode.UNAVAILABLE encountered while exporting metrics to mlops-collector-collector.monitoring-tools.svc.cluster.local:8889, retrying in 32s.


In [6]:
from opentelemetry import metrics
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.exporter.otlp.proto.grpc.metric_exporter import OTLPMetricExporter
from opentelemetry.sdk.metrics.export import PeriodicExportingMetricReader
import time

# -----------------------------
# CONFIGURAZIONE EXPORTER OTLP
# -----------------------------
exporter = OTLPMetricExporter(
    endpoint="mlops-collector-collector.monitoring-tools.svc.cluster.local:8889",
    insecure=True   # importantissimo su OpenShift
)

reader = PeriodicExportingMetricReader(exporter)

provider = MeterProvider(metric_readers=[reader])
metrics.set_meter_provider(provider)

meter = metrics.get_meter("training-metrics")

print("Exporter configurato correttamente ✓")

# ---------------------------------------------------
# 1) METRICA: training_accuracy (simulata come gauge)
# ---------------------------------------------------
accuracy_metric = meter.create_up_down_counter(
    name="training_accuracy",
    unit="1",
    description="training accuracy"
)

last_accuracy = 0.0

def set_accuracy(val):
    """
    Simula una gauge: imposta il valore attuale della accuracy
    """
    global last_accuracy
    accuracy_metric.add(val - last_accuracy)
    last_accuracy = val


# ---------------------------------------------------
# 2) METRICA: training_loss (simulata come gauge)
# ---------------------------------------------------
loss_metric = meter.create_up_down_counter(
    name="training_loss",
    unit="1",
    description="training loss"
)

last_loss = 0.0

def set_loss(val):
    """
    Simula una gauge: imposta il valore attuale della loss
    """
    global last_loss
    loss_metric.add(val - last_loss)
    last_loss = val


# -----------------------------------------
# 3) METRICA: training_step (semplice counter)
# -----------------------------------------
step_metric = meter.create_up_down_counter(
    name="training_step",
    unit="1",
    description="current training step"
)

current_step = 0

def next_step():
    global current_step
    current_step += 1
    step_metric.add(1)


# -------------------------
# SIMULAZIONE DEL TRAINING
# -------------------------
print("Invio metriche al Collector OTLP gRPC...\n")

accuracy_values = [0.1, 0.2, 0.35, 0.50, 0.65, 0.80]
loss_values =     [1.0, 0.7, 0.5, 0.35, 0.28, 0.20]

for acc, loss in zip(accuracy_values, loss_values):
    print(f"step {current_step+1}: accuracy={acc}, loss={loss}")

    next_step()
    set_accuracy(acc)
    set_loss(loss)

    time.sleep(3)  # tempo per permettere l'export

print("\nTraining completato ✓")
print("Controlla in OpenShift → Observe → Metrics: training_accuracy, training_loss, training_step")


Overriding of current MeterProvider is not allowed
An instrument with name training_accuracy, type UpDownCounter, unit 1 and description training accuracy has been created already.
An instrument with name training_loss, type UpDownCounter, unit 1 and description training loss has been created already.
An instrument with name training_step, type UpDownCounter, unit 1 and description current training step has been created already.


Exporter configurato correttamente ✓
Invio metriche al Collector OTLP gRPC...

step 1: accuracy=0.1, loss=1.0
step 2: accuracy=0.2, loss=0.7
step 3: accuracy=0.35, loss=0.5
step 4: accuracy=0.5, loss=0.35
step 5: accuracy=0.65, loss=0.28
step 6: accuracy=0.8, loss=0.2

Training completato ✓
Controlla in OpenShift → Observe → Metrics: training_accuracy, training_loss, training_step


Transient error StatusCode.UNAVAILABLE encountered while exporting metrics to mlops-collector-collector.monitoring-tools.svc.cluster.local:8889, retrying in 1s.
Transient error StatusCode.UNAVAILABLE encountered while exporting metrics to mlops-collector-collector.monitoring-tools.svc.cluster.local:8889, retrying in 2s.
Transient error StatusCode.UNAVAILABLE encountered while exporting metrics to mlops-collector-collector.monitoring-tools.svc.cluster.local:8889, retrying in 4s.
Transient error StatusCode.UNAVAILABLE encountered while exporting metrics to mlops-collector-collector.monitoring-tools.svc.cluster.local:8889, retrying in 8s.
Transient error StatusCode.UNAVAILABLE encountered while exporting metrics to mlops-collector-collector.monitoring-tools.svc.cluster.local:8889, retrying in 16s.
Transient error StatusCode.UNAVAILABLE encountered while exporting metrics to mlops-collector-collector.monitoring-tools.svc.cluster.local:8889, retrying in 32s.
Transient error StatusCode.UNAVA